# 06 · Multiple Datasets: Joint GNSS + InSAR Inversion

GNSS and InSAR see the same earthquake very differently. GNSS gives sparse but
full 3-D vectors at a few stations; InSAR gives a *dense* map of ground motion
but only along one line-of-sight direction. Together they constrain slip far
better than either alone. This notebook combines them in one inversion.

## Learning objectives
- Understand the InSAR **line-of-sight (LOS)** projection and look vectors.
- Stack multiple datasets into one linear system.
- Run a **joint** inversion with `geodef.invert(fault, [gnss, insar])`.
- See how relative data weighting shifts the solution.

## Why join datasets?

| | GNSS | InSAR |
| --- | --- | --- |
| sampling | sparse (few stations) | dense (many pixels) |
| components | full 3-D (E, N, U) | 1-D line-of-sight |
| reference | absolute | relative |

Their strengths are complementary: GNSS pins down the 3-D motion at a few points;
InSAR fills in the dense spatial pattern.

### The line-of-sight projection
A radar satellite measures only the component of ground motion **along its
viewing direction**. If the 3-D surface displacement is $\mathbf u = (u_e, u_n, u_u)$
and the unit look vector is $\hat{\mathbf l} = (l_e, l_n, l_u)$, the observation is the
scalar

$$ d_{\text{LOS}} = \mathbf u \cdot \hat{\mathbf l} = l_e u_e + l_n u_n + l_u u_u. $$

`geodef.InSAR` stores the per-pixel look vector and applies this projection when
building $G$, exactly as `GNSS` interleaves E/N/U rows.

### Stacking
A joint inversion just stacks the two forward problems vertically:

$$ \begin{bmatrix} G_{\text{GNSS}} \\ G_{\text{InSAR}} \end{bmatrix} \mathbf m
   = \begin{bmatrix} \mathbf d_{\text{GNSS}} \\ \mathbf d_{\text{InSAR}} \end{bmatrix}, $$

with a block-diagonal data covariance. Each dataset's own uncertainties set its
weight, so the more precise data naturally count more.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# A small planar fault, coarsely discretized so G is small and every inversion
# is instant. This same block is reused (copied) across notebooks 03-09.
fault = geodef.Fault.planar(
    lat=0.0, lon=100.0, depth=15e3,
    strike=90.0, dip=30.0,
    length=80e3, width=50e3,
    n_length=8, n_width=5,
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: a smooth Gaussian bump, dip-slip only. The model vector is
# blocked as [strike-slip (N) | dip-slip (N)], so we zero the strike-slip half.
i = np.arange(N) % nL
j = np.arange(N) // nL
bump = np.exp(-(((i - (nL - 1) / 2) / 2.0) ** 2 + ((j - (nW - 1) / 2) / 1.2) ** 2))
slip_true = np.concatenate([np.zeros(N), 2.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(99.5, 100.5, 6), np.linspace(-0.4, 0.4, 6))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.greens(
    fault, geodef.GNSS(glon, glat, _zero, _zero, _zero, _one, _one, _one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.GNSS(
    glon, glat,
    ve=d_obs[0::3], vn=d_obs[1::3], vu=d_obs[2::3],
    se=np.full(n_sta, sigma), sn=np.full(n_sta, sigma), su=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")


40 patches, 36 stations, 108 observations


In [2]:
# Build a synthetic InSAR track over the same fault and true slip.
# InSAR measures one number per pixel: ground motion projected onto the
# satellite line-of-sight (LOS) direction, l_hat = (look_e, look_n, look_u).
ilon, ilat = np.meshgrid(np.linspace(99.5, 100.5, 20), np.linspace(-0.4, 0.4, 20))
ilon, ilat = ilon.ravel(), ilat.ravel()
n_px = ilon.size

# A typical ascending-track look vector (mostly up, slightly east/north).
look_e = np.full(n_px, -0.38)
look_n = np.full(n_px, 0.09)
look_u = np.full(n_px, 0.92)

# Forward-model the true slip into LOS, then add correlated-free noise.
Gi = geodef.greens.greens(
    fault, geodef.InSAR(ilon, ilat, np.zeros(n_px), np.ones(n_px),
                        look_e, look_n, look_u)
)
sigma_los = 0.005  # 5 mm
los = Gi @ slip_true + rng.normal(0.0, sigma_los, n_px)
insar = geodef.InSAR(ilon, ilat, los=los, sigma=np.full(n_px, sigma_los),
                     look_e=look_e, look_n=look_n, look_u=look_u)
print(f"InSAR track: {n_px} pixels")


InSAR track: 400 pixels


## The two views of the same slip

GNSS vectors (left) and the InSAR LOS field (right). Note how few GNSS stations
there are compared with the dense InSAR pixels.

In [3]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
geodef.plot.vectors(gnss, fault, ax=ax1, title="GNSS displacements")
geodef.plot.insar(insar, fault, ax=ax2, title="InSAR line-of-sight")
plt.tight_layout()

## Single-dataset vs. joint inversion

Invert GNSS alone, InSAR alone, and both together (same smoothing for all).
The joint solution combines the constraints.

In [4]:
kw = dict(components="dip", smoothing="laplacian", smoothing_strength=1.0)
r_gnss = geodef.invert(fault, gnss, **kw)
r_insar = geodef.invert(fault, insar, **kw)
r_joint = geodef.invert(fault, [gnss, insar], **kw)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
vmax = slip_true[N:].max()
for ax, (name, r) in zip(
    axes,
    [("True", None), ("GNSS only", r_gnss),
     ("InSAR only", r_insar), ("Joint", r_joint)],
):
    m = slip_true[N:] if r is None else r.slip_vector
    geodef.plot.slip(fault, m, ax=ax, vmin=0, vmax=vmax, colorbar=False, title=name)
plt.tight_layout()

## Per-dataset fit

For a joint inversion, `dataset_diagnostics` reports the fit to each dataset
separately, so you can check that neither is being ignored.

In [5]:
for name, diag in zip(["GNSS", "InSAR"],
                      geodef.dataset_diagnostics(r_joint, fault, [gnss, insar])):
    print(f"{name}: reduced chi^2 = {diag.reduced_chi2:.2f}  "
          f"({diag.n_obs} obs, WRMS {diag.wrms * 1000:.1f} mm)")

GNSS: reduced chi^2 = 0.88  (108 obs, WRMS 935.3 mm)
InSAR: reduced chi^2 = 1.04  (400 obs, WRMS 977.4 mm)


## Exercises
1. **Down-weight InSAR.** Multiply `insar`'s `sigma` by 10 (rebuild the dataset)
   and re-run the joint inversion. Which way does the slip migrate?
2. **Second track.** Build a *descending* InSAR track (flip the sign of
   `look_e`) and add it as a third dataset. Does the extra look direction help?
3. **Coverage.** Shrink the GNSS grid to 3×3. How much does the dense InSAR
   compensate?

## Checkpoint questions
1. Why can InSAR alone leave slip poorly constrained even though it has many pixels?
2. What sets the relative weight of GNSS vs. InSAR in the joint system?
3. How does the look vector enter the Green's matrix rows for InSAR?

## Summary
- InSAR observes a scalar LOS projection $\mathbf u \cdot \hat{\mathbf l}$.
- Joint inversion stacks datasets with a block-diagonal covariance.
- `invert(fault, [gnss, insar])` and `dataset_diagnostics` handle and report both.

**Next:** notebook 07 tackles *correlated* InSAR noise, where the diagonal-$C_d$
assumption breaks down.